# Домашнее задание №2

**Максимальная оценка за работу `min(X, 10)`, где X - суммарный балл за hw2 из 12 возможных.**

## Часть 1. ML workflow (**всего 5 баллов**)

In [20]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

from sklearn.metrics import mean_squared_error, r2_score

from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

from itertools import product

### Загрузим данные для работы.

In [21]:
df = pd.read_csv(r"C:\Users\montl\Downloads\winequality-red.csv")
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


Будем решать задачу регрессии: необходимо предсказать качество вина на основе его характеристик

### Шаг 1.  (**0.2 балла**)
Создайте матрицу X объект-признак и целевой вектор y ("quality")

In [22]:
y = df['quality']
X = df.drop(columns=['quality'])

### Шаг 2. (**0.2 балла**)
Разбейте данные на train и test (доля тестовых данных - 30%).

In [23]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

### Шаг 3. (**0.2 балла**)
Обучите линейную регрессию на тренировочных данных и сделайте предсказания на train и на test.

In [24]:
model = LinearRegression()
model.fit(X_train, y_train)

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

### Шаг 4. (**0.4 балла**)
Выведите на экран ошибку MSE на train и на test, затем выведите на экран ошибку r2 на train и test.  

In [25]:
# MSE 
mse_train = mean_squared_error(y_train, train_pred)
mse_test = mean_squared_error(y_test, test_pred)

# R^2 
r2_train = r2_score(y_train, train_pred)
r2_test = r2_score(y_test, test_pred)

print("MSE на обучающей выборке", mse_train)
print("MSE на тестовой выборке", mse_test, "\n")
print("R^2 на обучающей выборке", r2_train)
print("R^2 на тестовой выборке", r2_test)

MSE на обучающей выборке 0.42078664921236153
MSE на тестовой выборке 0.41123487175042056 

R^2 на обучающей выборке 0.36119824413213175
R^2 на тестовой выборке 0.35138853325052266


### Шаг 5. (**0.5 балла**)
Вычислите среднее качество (r2) модели на кросс-валидации с k=5 фолдами.

In [26]:
cross_val_score(model, X_train, y=y_train, scoring = 'r2', cv = 5, n_jobs = -1).mean()

0.334985992062551

### Шаг 6.  (**0.5 балла**)
Теперь примените линейную регрессию с L1-регуляризацией (Lasso) для данной задачи. Объявите модель и подберите параметр регуляризации alpha по сетке. Ищите alpha в диапазоне (0.1, 1.1) с шагом 0.1.

Осуществите подбор параметра alpha по тренировочным данным (Xtrain, ytrain).

In [27]:
from sklearn.linear_model import Lasso

lasso_model = Lasso()
grid_parameters = {'alpha': np.linspace(0.1, 1.0, 10)}
grid_search = GridSearchCV(lasso_model, grid_parameters)
best_alpha = grid_search.fit(X_train, y_train).best_params_
print(best_alpha)

{'alpha': 0.1}


### Шаг 7.  (**0.5 балла**)
Выведите наилучший алгоритм и наилучшее качество по результатам подбора alpha (best_estimator_ и best_score_).

In [28]:
print("best estimator:", grid_search.best_estimator_, "\n")
print("best score:", grid_search.best_score_)

best estimator: Lasso(alpha=0.1) 

best score: 0.24002181024109742


### Шаг 8.  (**0.5 балла**)

С помощью найденного best_estimator_ сделайте предсказание на тестовых данных и выведите на экран r2-score на тесте.

In [29]:
model = grid_search.fit(X_train, y_train).best_estimator_
y_pr = model.predict(X_test)

print("r^2-score:", r2_score(y_test, y_pr))

r^2-score: 0.20887882853036954


### Шаг 9.  (**0.5 балла**)

Попробуем улучшить качество модели за счет добавления полиномиальных признаков. Создайте pipeline, состоящий из добавления полиномиальных признаков степени 2, а затем применения линейной регрессии.

Затем вычислите r2-score этой модели на кросс валидации с пятью фолдами.

In [30]:
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('poly_features', PolynomialFeatures(degree=2)),
    ('linear_regression', LinearRegression())])

r2_cv_score = cross_val_score(pipeline, X_train, y=y_train, scoring = 'r2', cv = 5, n_jobs = -1).mean()
print("Средний R2-score на кросс-валидации:", r2_cv_score)

Средний R2-score на кросс-валидации: 0.24380679870560865


### Шаг 10.  (**0.5 балла**)
Обучите модель (pipeline) на тренировочных данных и сделайте предсказания для train и test, затем выведите на экран r2-score и MSE на тренировочных и на тестовых данных.

In [31]:
pipeline.fit(X_train, y_train)
train_pred = pipeline.predict(X_train)
test_pred = pipeline.predict(X_test)

# MSE
mse_train = mean_squared_error(y_train, train_pred)
mse_test = mean_squared_error(y_test, test_pred)

# R^2
r2_train = r2_score(y_train, train_pred)
r2_test = r2_score(y_test, test_pred)

print("MSE на обучающей выборке:", mse_train)
print("MSE на тестовой выборке:", mse_test, "\n")
print("R^2 на обучающей выборке:", r2_train)
print("R^2 на тестовой выборке:", r2_test)

MSE на обучающей выборке: 0.3617804785173963
MSE на тестовой выборке: 0.411255683115978 

R^2 на обучающей выборке: 0.45077628924724655
R^2 на тестовой выборке: 0.35135570896623525


### Сделайте выводы. Для этого ответьте на вопросы: (**1 балл**)

1) Хорошее ли качество показала исходная модель (линейная регрессия без регуляризации)? Является ли эта модель переобученной?

2) Помогла ли L1-регуляризация улучшить качество модели?

3) Помогло ли добавление полиномов второй степени улучшить качество модели? Как добавление новых признаков повлияло на переобучение?

**1.** Показатели MSE и R² показывают, что модель не очень хороша, поскольку значения R² низкие, а ошибка MSE незначительная, но всё же существует. В тоже время разница между обучающей и тестовой выборкой мала, а значит переобучение маловероятно.

**2.** Показатель R^2 уменьшился с ~0.35 до 0.208, то есть качество модели ухудшилось.

**3.** Добавление полиномов второй степени привело к улучшению результатов относительно регуляризации, но в тоже время мы теперь наблюдаем признаки переобучения модели( разница между R^2 на выборках увеличилась).

### *Попытайтесь улучшить модель (добейтесь наилучшего качества) - можно использовать любые методы.

При улучшении качества r2 на 0.1-0.2 +1 балл, при большем улучшении +2 балла (дополнительно к 5 баллам за основную часть).

## Часть 2. Target encoding (**всего 5 баллов**)

В этом части домашнего задания вы будете работать с выборкой `1C`. Вам нужно посчитать счетчики для `item_id` четырьмя способами:

    1) При помощи KFold схемы;  
    2) При помощи Leave-one-out схемы;
    3) При помощи smoothing схемы;
    4) При помощи expanding mean схемы.

### Подготовка данных

In [32]:
sales = pd.read_csv(r"C:\Users\montl\Downloads\sales_train.csv.gz")
sales.columns = ['date', 'date_block_num', 'shop_id', 'item_id', 'item_price', 'target']
sales

,date,date_block_num,shop_id,item_id,item_price,target
0,02.01.2013,0,59,22154,999.00,1.0
1,03.01.2013,0,25,2552,899.00,1.0
2,05.01.2013,0,25,2552,899.00,-1.0
3,06.01.2013,0,25,2554,1709.05,1.0
4,15.01.2013,0,25,2555,1099.00,1.0
...,...,...,...,...,...,...
2935844,10.10.2015,33,25,7409,299.00,1.0
2935845,09.10.2015,33,25,7460,299.00,1.0
2935846,14.10.2015,33,25,7459,349.00,1.0
2935847,22.10.2015,33,25,7440,299.00,1.0


In [33]:
index_cols = ['shop_id', 'item_id', 'date_block_num']

# For every month we create a grid from all shops/items combinations from that month
grid = []
for block_num in sales['date_block_num'].unique():
    cur_shops = sales[sales['date_block_num']==block_num]['shop_id'].unique()
    cur_items = sales[sales['date_block_num']==block_num]['item_id'].unique()
    grid.append(np.array(list(product(*[cur_shops, cur_items, [block_num]])),dtype='int32'))

#turn the grid into pandas dataframe
grid = pd.DataFrame(np.vstack(grid), columns = index_cols,dtype=np.int32)

#get aggregated values for (shop_id, item_id, month)
gb = sales.groupby(index_cols,as_index=False).agg({'target':'sum'})

#join aggregated data to the grid
all_data = pd.merge(grid,gb,how='left',on=index_cols).fillna(0)
#sort the data
all_data.sort_values(['date_block_num','shop_id','item_id'],inplace=True)

### Mean encodings без регуляризации

После проделанной технической работы, мы готовы посчитать счетчики для переменной `item_id`.

Ниже приведены две реализации подсчета счетчиков без регуляризации. Можно использовать данный код в качестве стартовой точки для реализации различных техник регуляризации.

#### Способ 1

In [34]:
# Calculate a mapping: {item_id: target_mean}
item_id_target_mean = all_data.groupby('item_id').target.mean()

# In our non-regularized case we just *map* the computed means to the `item_id`'s
all_data['item_target_enc'] = all_data['item_id'].map(item_id_target_mean)

# Fill NaNs
all_data['item_target_enc'].fillna(0.3343, inplace=True)

# Print correlation
encoded_feature = all_data['item_target_enc'].values
print(np.corrcoef(all_data['target'].values, encoded_feature)[0][1])

0.48303869885778866


#### Способ 2

In [35]:
'''
     Differently to `.target.mean()` function `transform`
   will return a dataframe with an index like in `all_data`.
   Basically this single line of code is equivalent to the first two lines from of Method 1.
'''
all_data['item_target_enc'] = all_data.groupby('item_id')['target'].transform('mean')

# Fill NaNs
all_data['item_target_enc'].fillna(0.3343, inplace=True)

# Print correlation
encoded_feature = all_data['item_target_enc'].values
print(np.corrcoef(all_data['target'].values, encoded_feature)[0][1])

0.48303869885778866


###  KFold схема (**1.25 балла**)

Необходимо реализовать Kfold схему с пятью фолдами. Используйте KFold(5) из sklearn.model_selection.

1. Разбейте данные на 5 фолдов при помощи `sklearn.model_selection.KFold` с параметром `shuffle=False`.
2. Проитерируйтесь по фолдам: используйте 4 обучающих фолда для подсчета средних значений таргета по `item_id` и заполните этими значениями валидационный фолд на каждой итерации.

Обратите внимание на **Способ 1** из примера. В частности, изучите, как работают функции map и pd.Series.map. Они довольно полезны во многих ситуациях.

In [36]:
from sklearn.model_selection import KFold 

all_data['fold_mean_encoding'] = np.nan
kf = KFold(n_splits=5, shuffle=False)

for fold_number, (train_idx, val_idx) in enumerate(kf.split(all_data)):
    mean_encoded = all_data.loc[train_idx].groupby('item_id')['target'].mean()
    all_data.loc[val_idx, 'fold_mean_encoding'] = all_data.loc[val_idx, 'item_id'].map(mean_encoded)
    
corr = np.corrcoef(all_data['target'], all_data['fold_mean_encoding'].fillna(all_data['target'].mean()))[0, 1]
print(round(corr, 4))

0.4166


Ожидаемый ответ 0.4165

### Leave-one-out схема (**1.25 балла**)

Необходимо реализовать leave-one-out схему . Учтите, если вы запустите код из первого задания, задав количество фолдов такое же как размер выборки, то вы, вероятно, получите правильный ответ, но ждать будете очень-очень долго.

Для более быстрой реализации подсчета среднего таргета на всех объектах, кроме одного, вы можете:

1. Вычислить суммарный таргет по всем объектам.
2. Вычесть таргет конкретного объекта и разделить результат на `n_objects - 1`.

Заметим, что пункт `1.` следует сделать для всех объектов. Также заметим, что пункт `2.` может быть реализован без циклов `for`.

Здесь может оказаться полезной функция .transform из **Способа 2** из примера.

In [37]:
all_data['total_target'] = all_data.groupby('item_id')['target'].transform('sum')
all_data['num_items'] = all_data.groupby('item_id')['target'].transform('size') - 1
all_data['loo_mean'] = (all_data['total_target'] - all_data['target']) / all_data['num_items']
all_data.drop(['total_target', 'num_items'], axis=1, inplace=True)

corr = np.corrcoef(all_data['target'], all_data['loo_mean'])[0, 1]
print(round(correlation_result, 4))

0.4804


Ожидаемый ответ 0.4803

### Smoothing (**1.25 балла**)

Необходимо реализовать smoothing с $\alpha = 100$. Используйте формулу:

$\frac{mean(target) \cdot nrows + globalmean \cdot \alpha }{nrows + \alpha}$,

где $globalmean=0.3343$. Заметим, что `nrows` - это количество объектов, принадлежащих конкретной категории, а не количество строк в датасете.

In [38]:
global_mean = 0.3343
alpha = 100

mean_target_per_item = all_data.groupby('item_id')['target'].transform('mean')
count_per_item = all_data.groupby('item_id')['target'].transform('size')


all_data['smoothed_target'] = (mean_target_per_item * count_per_item + global_mean * alpha) / (count_per_item + alpha)


corr = np.corrcoef(all_data['target'].values, all_data['smoothed_target'].values)[0, 1]
print(round(corr, 4))

0.4818


Ожидаемый ответ 0.4818

### Expanding mean схема (**1.25 балла**)

Необходимо реализовать *expanding mean* схему. Ее суть заключается в том, чтобы пройти по отсортированному в определенном порядке датасету (датасет сортируется в самом начале задания) и для подсчета счетчика для строки $m$ использовать строки от $0$ до $m-1$. Вам будет необходимо воспользоваться pandas функциями [`cumsum`](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.core.groupby.DataFrameGroupBy.cumsum.html) и [`cumcount`](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.core.groupby.GroupBy.cumcount.html).

In [39]:
cumsum_target = all_data.groupby('item_id')['target'].cumsum() - all_data['target']
cumcount_target = all_data.groupby('item_id').cumcount()


all_data['expanding_mean'] = cumsum_target / cumcount_target.replace(0, np.nan)


corr = np.corrcoef(all_data['target'].values, all_data['expanding_mean'].fillna(all_data['target'].mean()))[0, 1]
print(round(corr, 4))

0.5025


Ожидаемый ответ 0.5025